# 消融
Stage 1；DPO；稳定性；完整模型；无状态更新；小 K；少量稳定性权重。

In [ ]:
from pathlib import Path
import os, sys, subprocess, zipfile
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = Path('/content/drive/MyDrive/SA-Jev')
    PROJECT = Path('/content/graphrag-jev')
except ImportError:
    DRIVE = Path(os.environ.get('SAJEV_WORKDIR', '../outputs/colab_local')).resolve()
    PROJECT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DRIVE.mkdir(parents=True, exist_ok=True)
REPO_URL = os.environ.get('SAJEV_REPO_URL', 'https://github.com/BoldHu/SA-Jev.git')  # Override with SAJEV_REPO_URL if needed.
SYNC_GIT = True
if not PROJECT.exists():
    if REPO_URL:
        subprocess.run(['git', 'clone', REPO_URL, str(PROJECT)], check=True)
    elif (DRIVE / 'sa-jev-source.zip').exists():
        PROJECT.mkdir(parents=True)
        with zipfile.ZipFile(DRIVE / 'sa-jev-source.zip') as archive:
            archive.extractall(PROJECT)
    else:
        raise RuntimeError('Set SAJEV_REPO_URL or copy sa-jev-source.zip to MyDrive/SA-Jev/')
if SYNC_GIT and (PROJECT / '.git').exists():
    status = subprocess.check_output(['git', '-C', str(PROJECT), 'status', '--porcelain'], text=True)
    if status.strip():
        print('Local edits present; preserving them. Git synchronization skipped.')
    else:
        subprocess.run(['git', '-C', str(PROJECT), 'pull', '--ff-only'], check=True)
os.chdir(PROJECT)
MODE = 'pilot'  # Change to 'full' only after the pilot completes and memory is measured.
ROOT = DRIVE / MODE
ROOT.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(DRIVE / 'huggingface')
print('Code:', PROJECT, 'Artifacts:', ROOT)


In [ ]:
# Run this in each fresh Colab runtime. No model weights are downloaded here.
subprocess.run([sys.executable, 'scripts/bootstrap_upstream.py'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[train,retrieval,test]'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-deps', '-e', 'vendor/OpenJev'], check=True)
if 'google.colab' in sys.modules:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'fla-core==0.5.2'], check=True)
# If package installation asks for a runtime restart, restart before importing torch.
from sa_jev.experiments import load_config, environment_manifest
config = load_config()
environment_manifest(ROOT / 'environment.json', config)
def cli(command, *args):
    subprocess.run([sys.executable, '-m', 'sa_jev.cli', command, '--root', str(ROOT), *map(str,args)], check=True)


In [ ]:
# Credentials are read from Colab Secrets or environment; never printed or saved.
if 'google.colab' in sys.modules:
    from google.colab import userdata
    for key in ('HF_TOKEN', 'OPENAI_API_KEY', 'OPENAI_BASE_URL', 'TYPESAFE_API_KEY'):
        if not os.environ.get(key):
            try:
                value = userdata.get(key)
                if value: os.environ[key] = value
            except Exception:
                pass


In [ ]:
from sa_jev.experiments import train_variant, main_experiment
from sa_jev.preparation import prepare_tasks
prepare_tasks(ROOT/'data', counts=config['data']['train_counts'], mine=True, small_k=True)
for seed in ([17] if MODE=='pilot' else config['seeds']):
    for variant in ['dpo','stability','small_stage1','small_k']:
        train_variant(ROOT, config, variant, seed, max_steps=2 if MODE=='pilot' else None, validation_limit=6 if MODE=='pilot' else None)
main_experiment(ROOT, config, table_name='ablations', methods=['openjev_frozen','stage1','dpo','stability','full','no_state_update','small_k'], seeds=[17] if MODE=='pilot' else config['seeds'], limit=8 if MODE=='pilot' else None)

权重选择只用 validation task loss/验证指标，不查看独立扰动测试来选权重。以下小扫掠先保留单种子。

In [ ]:
for weight in [0.05,0.2]:
    train_variant(ROOT, config, f'stab_weight_{weight}', 17, max_steps=2 if MODE=='pilot' else None, validation_limit=6 if MODE=='pilot' else None)